# Pipeline de transformation des données — Projet Spotify

Ce notebook transforme le dataset brut en **datasets dérivés** consommés par les
visualisations D3. Chaque section = une transformation qui lit un dataset et en
**produit un nouveau** (façon pipeline / notebook).

```
data/raw/spotify_songs.csv   (brut)
      │  00 nettoyage
      ▼
data/derived/tracks_clean.csv        (D0)
      │  01 normalisation        02 hiérarchie     03 agrégats     04 corrélation
      ▼                          ▼                 ▼               ▼
tracks_features.csv (D1)   hierarchy.json (D2)  trends.json (D3)  correlation.json (D4)
```

**Prérequis :** `pip install pandas numpy`
**Lancer :** ouvrir ce notebook depuis le dossier `scripts/` (Jupyter) et exécuter toutes les cellules.

In [1]:
import pandas as pd, numpy as np, json, os

In [2]:
# Chemins (relatifs au dossier scripts/)
RAW = "../data/raw/spotify_songs.csv"
OUT = "../data/derived"
os.makedirs(OUT, exist_ok=True)

# Features audio déjà dans [0,1]
FEATURES_01 = ["danceability","energy","valence","acousticness",
               "speechiness","instrumentalness","liveness"]
# Features à normaliser (échelles différentes)
FEATURES_NORM = ["tempo","loudness"]

## 00 · Nettoyage → `tracks_clean.csv` (D0)

- dédoublonnage sur `track_id` (un titre apparaît dans plusieurs playlists) ;
- `track_name` manquant → "Unknown Track" ;
- extraction de l'**année** depuis `track_album_release_date` (format `YYYY` ou `YYYY-MM-DD`) ;
- variables **calculées** : `decade`, `duration_min`, `mood = valence × energy`.

In [3]:
df = pd.read_csv(RAW)
print("brut :", len(df), "lignes")

df = df.drop_duplicates(subset="track_id").copy()
df["track_name"] = df["track_name"].fillna("Unknown Track")
df["year"]     = df["track_album_release_date"].astype(str).str.extract(r"(\d{4})").astype(float)
df["decade"]   = (df["year"] // 10 * 10)
df["duration_min"] = df["duration_ms"] / 60000
df["mood"]     = df["valence"] * df["energy"]

clean_cols = (["track_id","track_name","track_artist","track_popularity",
               "playlist_genre","playlist_subgenre","year","decade","duration_min","mood"]
              + FEATURES_01 + FEATURES_NORM)
d0 = df[clean_cols].rename(columns={"playlist_genre":"genre",
                                    "playlist_subgenre":"subgenre",
                                    "track_popularity":"popularity",
                                    "track_artist":"artist"})
d0.to_csv(f"{OUT}/tracks_clean.csv", index=False)
print("D0 tracks_clean :", len(d0), "titres uniques")
d0.head(3)

brut : 32833 lignes
D0 tracks_clean : 28356 titres uniques


,track_id,track_name,artist,popularity,genre,subgenre,year,decade,duration_min,mood,danceability,energy,valence,acousticness,speechiness,instrumentalness,liveness,tempo,loudness
0,6f807x0ima9a1j3VPbc7VN,I Don't Care (with Justin Bieber) - Loud Luxur...,Ed Sheeran,66,pop,dance pop,2019.0,2010.0,3.2459,0.474488,0.748,0.916,0.518,0.1020,0.0583,0.000000,0.0653,122.036,-2.634
1,0r7CVbZTWZgbTCYdfa2P31,Memories - Dillon Francis Remix,Maroon 5,67,pop,dance pop,2019.0,2010.0,2.7100,0.564795,0.726,0.815,0.693,0.0724,0.0373,0.004210,0.3570,99.972,-4.969
2,1z1Hg7Vb0AhHDiEmnDE79l,All the Time - Don Diablo Remix,Zara Larsson,70,pop,dance pop,2019.0,2010.0,2.9436,0.570703,0.675,0.931,0.613,0.0794,0.0742,0.000023,0.1100,124.008,-3.432


## 01 · Normalisation → `tracks_features.csv` (D1)

Pour les **coordonnées parallèles** (multivarié), toutes les features doivent être
comparables : on normalise `tempo` et `loudness` en [0,1] (min-max). Les autres
features sont déjà dans [0,1].

In [4]:
d1 = d0.copy()
for f in FEATURES_NORM:
    lo, hi = d1[f].min(), d1[f].max()
    d1[f + "_n"] = (d1[f] - lo) / (hi - lo)

feat_cols = (["track_id","track_name","artist","genre","subgenre","popularity"]
             + FEATURES_01 + ["tempo_n","loudness_n"])
d1[feat_cols].round(4).to_csv(f"{OUT}/tracks_features.csv", index=False)
print("D1 tracks_features :", len(d1), "titres x", len(FEATURES_01)+2, "features normalisées")

D1 tracks_features : 28356 titres x 9 features normalisées


## 02 · Hiérarchie → `hierarchy.json` (D2)

Arbre `genre → sous-genre → artiste` (format `d3.hierarchy`), la valeur d'une
feuille = **nombre de titres** de l'artiste. Alimente la **treemap / sunburst**.

In [5]:
def build_hierarchy(data):
    root = {"name": "Spotify", "children": []}
    for genre, g in data.groupby("genre"):
        gnode = {"name": genre, "children": []}
        for sub, s in g.groupby("subgenre"):
            counts = s.groupby("artist").size().reset_index(name="value")
            anodes = [{"name": a, "value": int(v)}
                      for a, v in zip(counts["artist"], counts["value"])]
            gnode["children"].append({"name": sub, "children": anodes})
        root["children"].append(gnode)
    return root

hier = build_hierarchy(d0)
with open(f"{OUT}/hierarchy.json", "w") as f:
    json.dump(hier, f, ensure_ascii=False)
print("D2 hierarchy :", len(hier["children"]), "genres")

D2 hierarchy : 6 genres


## 03 · Agrégats temporels → `trends.json` (D3)

Moyenne de chaque feature + nombre de titres, par **décennie × genre**. Alimente
les **courbes multiples**.

> ⚠️ Le dataset est très concentré sur 2019 et pauvre avant 1985 → on agrège par
> décennie, et on le signalera dans les limites du rapport.

In [6]:
agg_feats = FEATURES_01 + FEATURES_NORM + ["popularity"]
trends = (d0.dropna(subset=["decade"])
            .groupby(["decade","genre"])
            .agg(count=("track_id","size"), **{f:(f,"mean") for f in agg_feats})
            .reset_index())
trends["decade"] = trends["decade"].astype(int)
trends.round(4).to_json(f"{OUT}/trends.json", orient="records")
print("D3 trends :", len(trends), "lignes (décennie x genre)")

D3 trends : 43 lignes (décennie x genre)


## 04 · Matrice de corrélation → `correlation.json` (D4)  ·  *heatmap*

Corrélation de **Pearson** entre toutes les features (+ popularité), calculée sur
tous les titres. C'est une variable **calculée** (non stockée). Alimente la
**heatmap de corrélation**.

In [7]:
corr_feats = FEATURES_01 + FEATURES_NORM + ["popularity"]
corr = d0[corr_feats].corr(method="pearson").round(3)

d4 = {"features": corr_feats,
      "labels": {f: f.capitalize() for f in corr_feats},
      "matrix": corr.values.tolist()}
with open(f"{OUT}/correlation.json", "w") as f:
    json.dump(d4, f, ensure_ascii=False, indent=1)

print("D4 correlation :", len(corr_feats), "x", len(corr_feats))
print("  energy ~ loudness      =", corr.loc["energy","loudness"])
print("  energy ~ acousticness  =", corr.loc["energy","acousticness"])
corr

D4 correlation : 10 x 10
  energy ~ loudness      = 0.682
  energy ~ acousticness  = -0.546


,danceability,energy,valence,acousticness,speechiness,instrumentalness,liveness,tempo,loudness,popularity
danceability,1.000,-0.081,0.334,-0.029,0.183,-0.002,-0.127,-0.185,0.015,0.047
energy,-0.081,1.000,0.150,-0.546,-0.029,0.024,0.164,0.152,0.682,-0.104
valence,0.334,0.150,1.000,-0.019,0.065,-0.174,-0.020,-0.025,0.050,0.023
acousticness,-0.029,-0.546,-0.019,1.000,0.025,-0.003,-0.075,-0.114,-0.372,0.092
speechiness,0.183,-0.029,0.065,0.025,1.000,-0.108,0.059,0.033,0.013,0.005
instrumentalness,-0.002,0.024,-0.174,-0.003,-0.108,1.000,-0.009,0.021,-0.154,-0.124
liveness,-0.127,0.164,-0.020,-0.075,0.059,-0.009,1.000,0.022,0.082,-0.053
tempo,-0.185,0.152,-0.025,-0.114,0.033,0.021,0.022,1.000,0.097,0.004
loudness,0.015,0.682,0.050,-0.372,0.013,-0.154,0.082,0.097,1.000,0.037
popularity,0.047,-0.104,0.023,0.092,0.005,-0.124,-0.053,0.004,0.037,1.000


## 05 · Pays d'origine → `countries.json` (D5)  ·  *bonus (TODO)*

Le dataset **n'a pas** de colonne pays. L'enrichissement `artiste → pays d'origine`
se fera via **MusicBrainz** (variable calculée). À réaliser après le cœur du projet
— on prépare juste la liste des artistes à enrichir.

In [8]:
artists = d0["artist"].dropna().unique()
d5 = {"status": "TODO",
      "method": "artiste -> pays d'origine via MusicBrainz (variable calculée)",
      "n_artists_a_enrichir": int(len(artists)),
      "by_country": []}
with open(f"{OUT}/countries.json", "w") as f:
    json.dump(d5, f, ensure_ascii=False, indent=1)
print("D5 countries : stub (", len(artists), "artistes à enrichir )")

D5 countries : stub ( 10692 artistes à enrichir )


## Récapitulatif

| Dataset dérivé | Vue qui le consomme |
|---|---|
| `tracks_features.csv` (D1) | Coordonnées parallèles |
| `hierarchy.json` (D2) | Treemap / sunburst |
| `trends.json` (D3) | Courbes multiples |
| `correlation.json` (D4) | Heatmap de corrélation |
| `countries.json` (D5) | Carte (bonus) |

Dernière transformation (côté D3) : le **mapping** de ces données vers les
**variables visuelles** de chaque technique.